# Lab 5: LSTM Text Classification



## Cell 1: Import Libraries


In [1]:
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.datasets import imdb
from tensorflow.keras.layers import Dense, Dropout, Embedding, GRU, LSTM, SimpleRNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer

np.random.seed(42)
tf.random.set_seed(42)


## Cell 2: Sample Dataset


In [2]:
texts = [
    "I love this product",
    "This is a bad movie",
    "Excellent book",
    "Worst experience ever",
    "I feel great",
    "I hate it",
    "I hate this movie but ending is good",
    "The movie was boring but visuals were amazing",
]

labels = np.array([1, 0, 1, 0, 1, 0, 1, 1])


## Cell 3: Tokenize and Pad Sequences


In [3]:
max_words = 1000
max_len = 15

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding="post")

print("Example padded sequence:")
print(padded_sequences[6])


Example padded sequence:
[ 2  6  3  4  7 20  5 21  0  0  0  0  0  0  0]


## Cell 4: Train-Test Split


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences,
    labels,
    test_size=0.25,
    random_state=42,
)


## Cell 5: Build and Train RNN Model


In [5]:
embedding_dim = 50

rnn_model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    SimpleRNN(32),
    Dropout(0.5),
    Dense(1, activation="sigmoid"),
])

rnn_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
rnn_model.summary()

rnn_history = rnn_model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=2,
    validation_data=(X_test, y_test),
)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 186ms/step - accuracy: 0.3333 - loss: 0.7094 - val_accuracy: 1.0000 - val_loss: 0.6485
Epoch 2/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.5000 - loss: 0.7391 - val_accuracy: 0.5000 - val_loss: 0.6893
Epoch 3/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.8333 - loss: 0.5918 - val_accuracy: 0.5000 - val_loss: 0.7387
Epoch 4/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.8333 - loss: 0.5567 - val_accuracy: 0.5000 - val_loss: 0.7850
Epoch 5/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 39ms/step - accuracy: 1.0000 - loss: 0.4469 - val_accuracy: 0.5000 - val_loss: 0.8348
Epoch 6/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step - accuracy: 0.8333 - loss: 0.4937 - val_accuracy: 0.5000 - val_loss: 0.9009
Epoch 7/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 1.0000 - loss: 0.3095 - val_accuracy: 0.5000 - val_loss: 0.9958
Epoch 8/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 1.0000 - loss: 0.3200 - val_accuracy: 0.0000e+00 - val_loss: 1.

## Cell 6: Test RNN Prediction


In [6]:
new_text = ["I hate this movie but the ending is very bad"]
seq = tokenizer.texts_to_sequences(new_text)
padded_seq = pad_sequences(seq, maxlen=max_len, padding="post")
pred = rnn_model.predict(padded_seq)

print(f"Sentiment score: {pred[0][0]:.2f}")
print("Positive" if pred[0][0] > 0.5 else "Negative")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 169ms/step
Sentiment score: 0.96
Positive


## Cell 7: Build and Train LSTM Model


In [7]:
lstm_model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    LSTM(64),
    Dropout(0.5),
    Dense(1, activation="sigmoid"),
])

lstm_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
lstm_model.summary()

lstm_history = lstm_model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=2,
    validation_data=(X_test, y_test),
)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 195ms/step - accuracy: 0.6667 - loss: 0.6768 - val_accuracy: 0.0000e+00 - val_loss: 0.7265
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.8333 - loss: 0.6603 - val_accuracy: 0.0000e+00 - val_loss: 0.7708
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8333 - loss: 0.6451 - val_accuracy: 0.0000e+00 - val_loss: 0.8250
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.8333 - loss: 0.6137 - val_accuracy: 0.0000e+00 - val_loss: 0.8904
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8333 - loss: 0.6050 - val_accuracy: 0.0000e+00 - val_loss: 0.9779
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8333 - loss: 0.5632 - val_accuracy: 0.0000e+00 - val_loss: 1.1007
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8333 - loss: 0.5077 - val_accuracy: 0.0000e+00 - val_loss: 1.2701
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.8333 - loss: 0.4464 - val_accurac

## Cell 8: Evaluate and Predict With LSTM


In [8]:
loss, accuracy = lstm_model.evaluate(X_test, y_test)
print(f"LSTM Test Accuracy: {accuracy * 100:.2f}%")

new_texts = [
    "I hate this movie but the ending is good",
    "Absolutely loved the story and characters",
]

seq = tokenizer.texts_to_sequences(new_texts)
padded_seq = pad_sequences(seq, maxlen=max_len, padding="post")
predictions = lstm_model.predict(padded_seq)

for text, pred in zip(new_texts, predictions):
    sentiment = "Positive" if pred[0] > 0.5 else "Negative"
    print(f"Text: {text}")
    print(f"Predicted sentiment: {sentiment}")
    print()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 331ms/step - accuracy: 0.0000e+00 - loss: 2.0684
LSTM Test Accuracy: 0.00%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 189ms/step
Text: I hate this movie but the ending is good
Predicted sentiment: Positive

Text: Absolutely loved the story and characters
Predicted sentiment: Positive



## Cell 9: Build and Train GRU Model


In [9]:
gru_model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    GRU(64),
    Dropout(0.5),
    Dense(1, activation="sigmoid"),
])

gru_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
gru_model.summary()

gru_history = gru_model.fit(
    X_train,
    y_train,
    epochs=20,
    batch_size=2,
    validation_data=(X_test, y_test),
)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru (GRU)                       │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 311ms/step - accuracy: 0.5000 - loss: 0.6839 - val_accuracy: 0.0000e+00 - val_loss: 0.7319
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.8333 - loss: 0.6709 - val_accuracy: 0.0000e+00 - val_loss: 0.7818
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step - accuracy: 0.8333 - loss: 0.6163 - val_accuracy: 0.0000e+00 - val_loss: 0.8373
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.8333 - loss: 0.6199 - val_accuracy: 0.0000e+00 - val_loss: 0.9019
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step - accuracy: 0.8333 - loss: 0.5939 - val_accuracy: 0.0000e+00 - val_loss: 0.9745
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.8333 - loss: 0.5474 - val_accuracy: 0.0000e+00 - val_loss: 1.0543
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.8333 - loss: 0.5552 - val_accuracy: 0.0000e+00 - val_loss: 1.1427
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 77ms/step - accuracy: 0.8333 - loss: 0.4766 - val_accurac

## Cell 10: Evaluate and Predict With GRU


In [10]:
loss, accuracy = gru_model.evaluate(X_test, y_test)
print(f"GRU Test Accuracy: {accuracy * 100:.2f}%")

predictions = gru_model.predict(padded_seq)

for text, pred in zip(new_texts, predictions):
    sentiment = "Positive" if pred[0] > 0.5 else "Negative"
    print(f"Text: {text}")
    print(f"Predicted sentiment: {sentiment}")
    print()


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 411ms/step - accuracy: 0.0000e+00 - loss: 1.8061
GRU Test Accuracy: 0.00%
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 254ms/step
Text: I hate this movie but the ending is good
Predicted sentiment: Positive

Text: Absolutely loved the story and characters
Predicted sentiment: Positive



## Cell 11: Compare RNN, LSTM, and GRU


In [13]:
print("=== FINAL COMPARISON ===")

print(f"RNN  -> Train Acc: {rnn_history.history['accuracy'][-1]:.4f}, "
      f"Loss: {rnn_history.history['loss'][-1]:.4f}, "
      f"Val Acc: {rnn_history.history['val_accuracy'][-1]:.4f}, "
      f"Val Loss: {rnn_history.history['val_loss'][-1]:.4f}")

print(f"LSTM -> Train Acc: {lstm_history.history['accuracy'][-1]:.4f}, "
      f"Loss: {lstm_history.history['loss'][-1]:.4f}, "
      f"Val Acc: {lstm_history.history['val_accuracy'][-1]:.4f}, "
      f"Val Loss: {lstm_history.history['val_loss'][-1]:.4f}")

print(f"GRU  -> Train Acc: {gru_history.history['accuracy'][-1]:.4f}, "
      f"Loss: {gru_history.history['loss'][-1]:.4f}, "
      f"Val Acc: {gru_history.history['val_accuracy'][-1]:.4f}, "
      f"Val Loss: {gru_history.history['val_loss'][-1]:.4f}")


=== FINAL COMPARISON ===
RNN  -> Train Acc: 1.0000, Loss: 0.0433, Val Acc: 0.0000, Val Loss: 2.5678
LSTM -> Train Acc: 0.8333, Loss: 0.3425, Val Acc: 0.0000, Val Loss: 2.0684
GRU  -> Train Acc: 0.8333, Loss: 0.4852, Val Acc: 0.0000, Val Loss: 1.8061


## Cell 12: Load IMDB Dataset


In [14]:
max_words = 20000

(X_train_imdb, y_train_imdb), (X_test_imdb, y_test_imdb) = imdb.load_data(num_words=max_words)

print("Training samples:", len(X_train_imdb))
print("Testing samples:", len(X_test_imdb))


17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Training samples: 25000
Testing samples: 25000


## Cell 13: Pad IMDB Sequences


In [15]:
max_len = 300

X_train_imdb = pad_sequences(X_train_imdb, maxlen=max_len)
X_test_imdb = pad_sequences(X_test_imdb, maxlen=max_len)


## Cell 14: Build IMDB LSTM Model


In [16]:
imdb_model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(128),
    Dropout(0.3),
    Dense(1, activation="sigmoid"),
])

imdb_model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])
imdb_model.summary()


Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

## Cell 15: Train IMDB LSTM Model


In [17]:
early_stop = EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True)

imdb_history = imdb_model.fit(
    X_train_imdb,
    y_train_imdb,
    epochs=10,
    batch_size=64,
    validation_split=0.2,
    callbacks=[early_stop],
)


Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 256s 808ms/step - accuracy: 0.7812 - loss: 0.4611 - val_accuracy: 0.8662 - val_loss: 0.3317
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 251s 801ms/step - accuracy: 0.8885 - loss: 0.2796 - val_accuracy: 0.8582 - val_loss: 0.3387
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 249s 795ms/step - accuracy: 0.9072 - loss: 0.2423 - val_accuracy: 0.8616 - val_loss: 0.3656


## Cell 16: Evaluate IMDB LSTM Model


In [ ]:
loss, accuracy = imdb_model.evaluate(X_test_imdb, y_test_imdb)
print(f"IMDB Test Accuracy: {accuracy * 100:.2f}%")
